# Multi-factor research pipeline

配置化的多因子研究流程：数据准备、因子构造、横截面变换和评估彼此解耦。新增因子时只需扩展 `FACTOR_SPECS`，无需复制标准化、分组或绘图代码。

In [ ]:
from __future__ import annotations

from collections.abc import Callable, Sequence
from dataclasses import dataclass
from datetime import timedelta
from math import ceil

import matplotlib.pyplot as plt
import polars as pl

from trend_trader.data.models import bar_minutes

pl.Config.set_tbl_rows(10)

## 1. Configuration and factor registry

In [ ]:
@dataclass(frozen=True)
class AnalysisConfig:
    candles_path: str = (
        "../data/market/v1/offline/normalized/candles/"
        "venue=BINANCE/year=*/date=*/*.parquet"
    )
    funding_rates_path: str = (
        "../data/market/v1/offline/normalized/funding_rates/"
        "venue=BINANCE/year=*/date=*/*.parquet"
    )
    market_type: str = "UM"
    bar_frequency: str = "1h"
    volume_lookback: int = 30
    universe_size: int = 30
    breakout_window: int = 20
    momentum_windows: tuple[int, ...] = (10, 6, 12, 24, 36)
    return_horizon: int = 1
    quantiles: int = 10
    evaluate_universe_only: bool = True


@dataclass(frozen=True)
class FactorSpec:
    name: str
    expression: Callable[[AnalysisConfig], pl.Expr]
    label: str
    make_quantiles: bool = True


def momentum_expression(window: int) -> Callable[[AnalysisConfig], pl.Expr]:
    def expression(_: AnalysisConfig) -> pl.Expr:
        previous_close = pl.col("close").shift(window)
        return (pl.col("close") / previous_close - 1).shift(1).over("instrument_id")

    return expression


def breakout_expression(config: AnalysisConfig) -> pl.Expr:
    window = config.breakout_window
    center = (window - 1) / 2
    periods_since_high = pl.col("close").rolling_map(
        lambda values: len(values) - 1 - values.arg_max(),
        window_size=window,
    )
    return (center - periods_since_high).shift(1).over("instrument_id")


def carry_expression(_: AnalysisConfig) -> pl.Expr:
    return pl.col("funding_rate").shift(1).over("instrument_id")


def make_factor_specs(config: AnalysisConfig) -> tuple[FactorSpec, ...]:
    momentum_specs = tuple(
        FactorSpec(
            name="momo" if window == config.momentum_windows[0] else f"momo{window}",
            expression=momentum_expression(window),
            label=f"Momentum ({window}h)",
        )
        for window in config.momentum_windows
    )
    return (
        FactorSpec("breakout", breakout_expression, "Breakout"),
        *momentum_specs,
        FactorSpec("carry", carry_expression, "Carry"),
    )

## 2. Data pipeline

时间约定：`timestamp` 是 bar 开盘时间；因子和交易池只使用此前已经完成的 bar；目标收益采用 `open[t] → open[t+horizon]`；资金费率只在持有区间内的真实结算时点计入。

In [ ]:
def load_base_panel(config: AnalysisConfig) -> pl.DataFrame:
    candle_columns = [
        "timestamp", "instrument_id", "market_type",
        "open", "high", "low", "close", "volume_quote",
    ]
    candles = (
        pl.scan_parquet(config.candles_path)
        .select(candle_columns)
        .filter(pl.col("market_type") == config.market_type)
        .sort(["instrument_id", "timestamp"])
        .unique(
            subset=["instrument_id", "timestamp"],
            keep="last",
            maintain_order=True,
        )
        .group_by_dynamic(
            "timestamp",
            every=config.bar_frequency,
            group_by="instrument_id",
            closed="left",
            label="left",
        )
        .agg(
            pl.col("open").first(),
            pl.col("high").max(),
            pl.col("low").min(),
            pl.col("close").last(),
            pl.col("volume_quote").sum().alias("volume"),
        )
    )

    funding_rates = (
        pl.scan_parquet(config.funding_rates_path)
        .select(
            pl.col("funding_time").alias("timestamp"),
            "instrument_id",
            "funding_rate",
        )
        .sort(["instrument_id", "timestamp"])
        .unique(
            subset=["instrument_id", "timestamp"],
            keep="last",
            maintain_order=True,
        )
    )

    return (
        candles.join(funding_rates, on=["timestamp", "instrument_id"], how="left")
        .sort(["instrument_id", "timestamp"])
        .with_columns(
            # 资金费率是结算事件；现金流只保留在事件时点。
            pl.col("funding_rate").fill_null(0.0).alias("funding_cashflow"),
            # carry 信号使用最近一次已知费率，与结算现金流分开保存。
            pl.col("funding_rate").forward_fill().over("instrument_id"),
            # timestamp 是 bar 开盘时间，交易池只能使用上一根及更早的成交量。
            pl.col("volume")
            .rolling_mean(window_size=config.volume_lookback)
            .shift(1)
            .over("instrument_id")
            .alias("trail_volume"),
        )
        .filter(pl.col("trail_volume").is_not_null())
        .with_columns(
            pl.col("trail_volume")
            .rank(method="ordinal", descending=True)
            .over("timestamp")
            .alias("volume_rank")
        )
        .with_columns(
            (pl.col("volume_rank") <= config.universe_size).alias("is_universe")
        )
        .sort(["timestamp", "volume_rank"])
        .collect()
    )

In [ ]:
def add_forward_returns(frame: pl.DataFrame, config: AnalysisConfig) -> pl.DataFrame:
    horizon = config.return_horizon
    if horizon <= 0:
        raise ValueError("return_horizon must be positive")

    step = timedelta(minutes=bar_minutes(config.bar_frequency))
    future_open = pl.col("open").shift(-horizon).over("instrument_id")
    observed_exit = pl.col("timestamp").shift(-horizon).over("instrument_id")
    expected_exit = pl.col("timestamp") + step * horizon
    continuous = observed_exit == expected_exit
    price_return = future_open / pl.col("open") - 1
    funding_paid = pl.sum_horizontal(
        [
            pl.col("funding_cashflow").shift(-offset).over("instrument_id")
            for offset in range(1, horizon + 1)
        ]
    )

    return (
        frame.sort(["instrument_id", "timestamp"])
        .with_columns(
            expected_exit.alias("exit_time"),
            continuous.fill_null(False).alias("return_is_valid"),
            pl.when(continuous).then(funding_paid).alias("funding_paid"),
            pl.when(continuous).then(price_return).alias("rets_without_funding"),
            # 正资金费率由多头支付，因此从多头价格收益中扣除。
            pl.when(continuous)
            .then(price_return - funding_paid)
            .alias("rets_with_funding"),
        )
        .with_columns(
            pl.col("rets_without_funding").log1p().alias("log_rets_without_funding"),
            pl.col("rets_with_funding").log1p().alias("log_rets_with_funding"),
        )
    )


def add_factors(
    frame: pl.DataFrame,
    config: AnalysisConfig,
    factor_specs: Sequence[FactorSpec],
) -> pl.DataFrame:
    expressions = [spec.expression(config).alias(spec.name) for spec in factor_specs]
    factor_names = [spec.name for spec in factor_specs]
    return (
        frame.sort(["instrument_id", "timestamp"])
        .with_columns(expressions)
        .with_columns(
            [
                pl.when(pl.col(name).is_finite()).then(pl.col(name)).alias(name)
                for name in factor_names
            ]
        )
    )


def add_cross_sectional_transforms(
    frame: pl.DataFrame,
    config: AnalysisConfig,
    factor_specs: Sequence[FactorSpec],
) -> pl.DataFrame:
    return_columns = ["log_rets_with_funding", "log_rets_without_funding"]
    frame = frame.filter(
        pl.col("return_is_valid")
        & pl.all_horizontal([pl.col(column).is_finite() for column in return_columns])
    )
    if config.evaluate_universe_only:
        frame = frame.filter(pl.col("is_universe"))

    frame = frame.with_columns(
        [
            (pl.col(column) - pl.col(column).mean().over("timestamp"))
            .alias(f"demeaned_{column.removeprefix('log_')}")
            for column in return_columns
        ]
    )

    transforms: list[pl.Expr] = []
    for spec in factor_specs:
        factor = pl.col(spec.name)
        factor_std = factor.std().over("timestamp")
        valid_count = factor.count().over("timestamp")
        transforms.append(
            pl.when(factor_std > 0)
            .then((factor - factor.mean().over("timestamp")) / factor_std)
            .alias(f"z_{spec.name}")
        )
        if spec.make_quantiles:
            transforms.append(
                (
                    factor.rank(method="ordinal").over("timestamp")
                    * config.quantiles
                    / valid_count
                )
                .ceil()
                .clip(1, config.quantiles)
                .cast(pl.UInt8)
                .alias(f"quantile_{spec.name}")
            )

    return frame.with_columns(transforms)


def build_analysis_panel(
    config: AnalysisConfig, factor_specs: Sequence[FactorSpec]
) -> tuple[pl.DataFrame, pl.DataFrame]:
    base_panel = load_base_panel(config)
    factor_panel = add_factors(add_forward_returns(base_panel, config), config, factor_specs)
    analysis_panel = add_cross_sectional_transforms(factor_panel, config, factor_specs)
    return base_panel, analysis_panel

## 3. Reusable diagnostics and evaluation

In [ ]:
def plot_universe_history(frame: pl.DataFrame) -> None:
    counts = (
        frame.group_by(["timestamp", "is_universe"])
        .len(name="count")
        .sort("timestamp")
        .to_pandas()
    )
    for is_universe, label in [(True, "In universe"), (False, "Outside universe")]:
        subset = counts[counts["is_universe"] == is_universe]
        plt.plot(subset["timestamp"], subset["count"], label=label)
    plt.title("Universe size")
    plt.xlabel("Time")
    plt.ylabel("Count")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def plot_histograms(
    frame: pl.DataFrame, columns: Sequence[str], *, bins: int = 200, columns_per_row: int = 3
) -> None:
    row_count = ceil(len(columns) / columns_per_row)
    figure, axes = plt.subplots(
        row_count, columns_per_row, figsize=(4.3 * columns_per_row, 3 * row_count), squeeze=False
    )
    for axis, column in zip(axes.flat, columns, strict=False):
        values = frame.get_column(column).drop_nulls()
        axis.hist(values.filter(values.is_finite()).to_numpy(), bins=bins)
        axis.set(title=column, xlabel="Value", ylabel="Count")
    for axis in axes.flat[len(columns):]:
        axis.set_visible(False)
    figure.tight_layout()
    plt.show()


def quantile_return_table(
    frame: pl.DataFrame,
    factor_name: str,
    *,
    return_column: str = "demeaned_rets_without_funding",
) -> pl.DataFrame:
    quantile_column = f"quantile_{factor_name}"
    return (
        frame.filter(
            pl.col(quantile_column).is_not_null() & pl.col(return_column).is_finite()
        )
        .group_by(quantile_column)
        .agg(
            pl.col(return_column).mean().alias("mean_return"),
            pl.col(return_column).std().alias("return_std"),
            pl.len().alias("observations"),
        )
        .sort(quantile_column)
    )


def plot_quantile_returns(
    frame: pl.DataFrame, factor_specs: Sequence[FactorSpec], *, return_column: str
) -> None:
    specs = [spec for spec in factor_specs if spec.make_quantiles]
    figure, axes = plt.subplots(
        ceil(len(specs) / 3), 3, figsize=(13, 3 * ceil(len(specs) / 3)), squeeze=False
    )
    for axis, spec in zip(axes.flat, specs, strict=False):
        table = quantile_return_table(frame, spec.name, return_column=return_column)
        quantile_column = f"quantile_{spec.name}"
        axis.bar(table[quantile_column].to_numpy(), table["mean_return"].to_numpy())
        axis.set(title=spec.label, xlabel="Quantile", ylabel="Mean demeaned return")
    for axis in axes.flat[len(specs):]:
        axis.set_visible(False)
    figure.tight_layout()
    plt.show()

In [ ]:
def information_coefficient_summary(
    frame: pl.DataFrame,
    factor_specs: Sequence[FactorSpec],
    *,
    return_column: str = "demeaned_rets_without_funding",
) -> tuple[pl.DataFrame, pl.DataFrame]:
    factor_columns = [f"z_{spec.name}" for spec in factor_specs]
    ic_series = (
        frame.unpivot(
            index=["timestamp", return_column],
            on=factor_columns,
            variable_name="factor",
            value_name="factor_value",
        )
        .filter(
            pl.col("factor_value").is_finite()
            & pl.col(return_column).is_finite()
        )
        .group_by(["timestamp", "factor"])
        .agg(pl.corr("factor_value", return_column).alias("ic"))
        .drop_nulls("ic")
    )
    summary = (
        ic_series.group_by("factor")
        .agg(
            pl.col("ic").mean().alias("mean_ic"),
            pl.col("ic").std().alias("ic_std"),
            pl.len().alias("periods"),
        )
        .with_columns((pl.col("mean_ic") / pl.col("ic_std")).alias("ic_ir"))
        .with_columns(
            pl.col("factor").replace_strict(
                {name: index for index, name in enumerate(factor_columns)}
            ).alias("sort_order")
        )
        .sort("sort_order")
        .drop("sort_order")
    )
    return ic_series, summary


def plot_ic_summary(summary: pl.DataFrame) -> None:
    plt.figure(figsize=(10, 3))
    plt.bar(summary["factor"].to_list(), summary["mean_ic"].to_list())
    plt.xlabel("Factor")
    plt.ylabel("Mean cross-sectional IC")
    plt.title("Information coefficient by factor")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 4. Run analysis

In [ ]:
CONFIG = AnalysisConfig()
FACTOR_SPECS = make_factor_specs(CONFIG)

base_panel, analysis_panel = build_analysis_panel(CONFIG, FACTOR_SPECS)
analysis_panel

In [ ]:
plot_universe_history(base_panel)
plot_histograms(analysis_panel, [spec.name for spec in FACTOR_SPECS])
plot_histograms(analysis_panel, [f"z_{spec.name}" for spec in FACTOR_SPECS])

In [ ]:
plot_quantile_returns(
    analysis_panel,
    FACTOR_SPECS,
    return_column="demeaned_rets_without_funding",
)

In [ ]:
ic_series, ic_summary = information_coefficient_summary(analysis_panel, FACTOR_SPECS)
plot_ic_summary(ic_summary)
ic_summary

## 5. Add a factor

新增因子只需定义一个返回 Polars 表达式的函数，并注册到 `FACTOR_SPECS`。例如：

```python
def volatility_expression(_: AnalysisConfig) -> pl.Expr:
    return pl.col("close").pct_change().rolling_std(24).shift(1).over("instrument_id")

FACTOR_SPECS = (*FACTOR_SPECS, FactorSpec("volatility", volatility_expression, "Volatility (24h)"))
base_panel, analysis_panel = build_analysis_panel(CONFIG, FACTOR_SPECS)
```